# 03 · Simulator, safety shield, planners and controllers; training and validation episodes

1. Writes `src/sim.py` (street simulator + 4 h safety shield), `src/planners.py` (MILP) and `src/controllers.py` (rule-based, MPC v2,
   MPC-CC, perfect foresight, the expert). Physics, reward, state, shield and baseline controllers are **those of versions 1-3, unchanged**.
2. Builds the simulator bank (one array per region, forecasts from the frozen forecasters of notebook 02) and the episode lists.
3. **Regression check:** on the archived v2 validation episodes, the new code reproduces the archived results exactly
   (rule-based, rule-based + shield, MPC v2, perfect foresight, the expert, and the frozen PPO-BC policy).

In [1]:
%%writefile ../src/sim.py
"""Nirantar street simulator with the safety shield. Written from notebooks/03_simulator.ipynb.

Physics and reward are those of versions 1-3 (archived src/street_env.py), unchanged; the shield is the version 2 shield with a
4 h reserve (archived src/v2.py, duration model off). Only the data access differs: a "bank" holds one array per region, and an
episode is (region, first row, homes), so overlapping training windows share memory.

Physics per 15-minute step:
  1. Homes in a cut are fed from the shared backup line: solar first, then the battery. Essentials (30% of demand)
     first, shared in proportion; the rest of their load only as far as the backup share allows.
  2. Leftover solar feeds homes with grid supply; surplus charges the battery; the rest is curtailed (no export).
  3. The controller's battery action: discharge to grid-connected homes, or charge from the grid (never both in a step).
Shield (before every action): need = expected essential energy of the street over the next 4 h; avail = usable battery energy.
  1. discharge to grid-connected homes is reduced (1 -> 0.5 -> 0) until avail - discharge >= need
  2. non-essential load of homes in a cut -> essentials only if avail - that energy < need
  3. if avail < need, no home is in a cut and it is not 18-22 h: charge from the grid at full power
"""
from collections import deque
from dataclasses import dataclass, replace
from pathlib import Path
import json, pickle
import numpy as np
import gymnasium as gym
from gymnasium import spaces

DT, SPD, WEEK = 0.25, 96, 672
BATT_LEVELS = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])
NL = len(BATT_LEVELS)
N_OBS = 23
UNSERVED_KWH = 1e-3          # a home-step counts as "essentials off" if short by more than 1 Wh
WEAR_INR_PER_KWH = 1.18      # BNEF 2025 pack price x LFP cycle life (results/config.json)
HORIZON_H = 4.0


@dataclass
class StreetConfig:
    n_homes: int = 10
    pv_kwp_per_home: float = 0.5
    batt_kwh_per_home: float = 0.75
    batt_kw_per_home: float = 0.25
    eta: float = 0.95
    soc_min: float = 0.10
    essential_share: float = 0.30
    peak_hours: tuple = (18, 22)
    peak_cap_kw_per_home: float = 0.5
    w_ess: float = 10.0
    w_other: float = 3.0
    w_peak: float = 1.0
    w_peak_excess: float = 3.0
    w_wear: float = 0.20
    w_fair: float = 5.0
    reward_scale: float = 0.1
    use_forecasts: bool = True

    @property
    def pv_kwp(self): return self.pv_kwp_per_home * self.n_homes
    @property
    def batt_kwh(self): return self.batt_kwh_per_home * self.n_homes
    @property
    def batt_kw(self): return self.batt_kw_per_home * self.n_homes
    @property
    def cap_kw(self): return self.peak_cap_kw_per_home * self.n_homes


def load_cfg(root):
    c = dict(json.loads((Path(root) / "results" / "config.json").read_text())["config"])
    c["peak_hours"] = tuple(c["peak_hours"])
    return StreetConfig(**c)


def jain(x):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return 1.0
    if (x ** 2).sum() == 0:
        return float("nan")
    return float(x.sum() ** 2 / (len(x) * (x ** 2).sum()))


_BANKS = {}


def load_bank(path):
    """Region arrays (cached per process). bank[region] = {"start": first timestamp, "homes": [...], "demand": [T, N], "cut": [T, N] bool,
    "pv", "pv_obs", "pv_prev", "cut_rate": [T], "cut_p": [T, N, 3], "load_fc": [T, N, 3], optional "cut_p_ref", "load_fc_ref"}."""
    path = str(path)
    if path not in _BANKS:
        _BANKS[path] = pickle.loads(Path(path).read_bytes())
    return _BANKS[path]


def episode_arrays(bank, ep, cut_key="cut_p", load_key="load_fc"):
    """Arrays of one week for the episode's homes. ep = {"region", "t0", "pool_idx", optional "homes_idx"}."""
    B = bank[ep["region"]]; a = ep["t0"]; b = a + WEEK
    return {"demand": B["demand"][a:b], "cut": B["cut"][a:b], "pv": B["pv"][a:b], "pv_obs": B["pv_obs"][a:b], "pv_prev": B["pv_prev"][a:b],
            "cut_rate": B["cut_rate"][a:b], "cut_p": B[cut_key][a:b], "load_fc": B[load_key][a:b], "start": B["start"] + np.timedelta64(15 * a, "m")}


class StreetEnv(gym.Env):
    """obs: 23 inputs (v2 state). stack > 1 returns the last `stack` states concatenated (oldest first), for the GRU policy."""
    metadata = {"render_modes": []}

    def __init__(self, bank, episodes, cfg, train=True, seed=0, shield=True, stack=1, cut_key="cut_p", load_key="load_fc"):
        super().__init__()
        self.bank, self.episodes, self.cfg, self.train = bank, episodes, cfg, train
        self.use_shield, self.stack, self.cut_key, self.load_key = shield, stack, cut_key, load_key
        self.rng = np.random.default_rng(seed)
        self.action_space = spaces.Discrete(NL * 2)
        self.observation_space = spaces.Box(-1.0, 5.0, shape=(N_OBS * stack,), dtype=np.float32)
        self._i = -1
        self.shield_steps = 0; self.steps = 0

    # ---------- episodes ----------
    def _pick(self, options):
        if self.train:
            ep = self.episodes[self.rng.integers(len(self.episodes))]
            cols = np.asarray(ep["pool_idx"])[np.sort(self.rng.choice(len(ep["pool_idx"]), self.cfg.n_homes, replace=False))]
        else:
            self._i = options["episode"] if options and "episode" in options else (self._i + 1) % len(self.episodes)
            ep = self.episodes[self._i]; cols = np.asarray(ep["homes_idx"])
        return ep, cols

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        ep, cols = self._pick(options)
        A = episode_arrays(self.bank, ep, self.cut_key, self.load_key); c = self.cfg
        self.ep, self.cols = ep, cols
        self.demand = A["demand"][:, cols].astype(np.float64)
        self.off = A["cut"][:, cols].astype(bool)
        self.pv = A["pv"].astype(np.float64) * c.pv_kwp
        self.pv_obs = A["pv_obs"].astype(np.float64); self.pv_prev = A["pv_prev"].astype(np.float64)
        self.cut_rate = A["cut_rate"].astype(np.float64)
        self.cut_p = A["cut_p"][:, cols, :].astype(np.float64); self.load_fc = A["load_fc"][:, cols, :].astype(np.float64)
        self.start = A["start"]
        self.T = self.demand.shape[0]; self.t = 0
        self.soc = (self.rng.uniform(0.3, 0.9) if self.train else 0.5) * c.batt_kwh
        if options and "soc" in options:
            self.soc = float(options["soc"]) * c.batt_kwh
        n = c.n_homes
        self.cut_demand, self.cut_served = np.zeros(n), np.zeros(n)
        self.log = {k: [] for k in ["ess_uns", "oth_uns", "import", "pv_used", "pv_avail", "dis", "ch", "soc", "ess_uns_homes"]}
        self.shield_steps = 0; self.steps = 0
        o = self._obs_one(); self.hist = deque([o] * self.stack, maxlen=self.stack)
        return self._stacked(), {}

    def restart_history(self):
        """After changing the state by hand (e.g. the battery charge), rebuild the observation history from the current state."""
        o = self._obs_one(); self.hist = deque([o] * self.stack, maxlen=self.stack)
        return self._stacked()

    # ---------- observation (identical to v1-v3, 23 inputs) ----------
    def _obs_one(self):
        c, t = self.cfg, min(self.t, self.T - 1)
        d, off = self.demand[t], self.off[t]
        k = t % SPD
        dow = (int((self.start - np.datetime64("1970-01-05")) / np.timedelta64(1, "D")) + t // SPD) % 7
        tod = 2 * np.pi * k / SPD
        day0 = (t // SPD) * SPD
        cr = lambda h: self.cut_rate[day0 + (t + h) % SPD]
        served = self.cut_served / np.maximum(self.cut_demand, 1e-9)
        worst = served[self.cut_demand > 1e-6].min() if (self.cut_demand > 1e-6).any() else 1.0
        o = [self.soc / c.batt_kwh if c.batt_kwh > 0 else 0.0, self.pv_obs[t], self.pv_prev[t],
             d[~off].sum() / c.n_homes, c.essential_share * d[off].sum() / c.n_homes, d[off].sum() / c.n_homes, off.mean(),
             np.sin(tod), np.cos(tod), np.sin(2 * np.pi * dow / 7), np.cos(2 * np.pi * dow / 7),
             cr(0), cr(4), cr(8), worst]
        p, lf = self.cut_p[t], self.load_fc[t]                   # [homes, 3] for +1 h, +2 h, +4 h
        o += list(p.mean(0)) + list((p[:, :2] * c.essential_share * lf[:, :2]).mean(0)) + list(lf.mean(0))
        return np.asarray(o, dtype=np.float32)

    def _stacked(self):
        return np.concatenate(list(self.hist)).astype(np.float32) if self.stack > 1 else self.hist[-1]

    # ---------- shield (v2, 4 h, no duration model) ----------
    def need_avail(self):
        c, t = self.cfg, self.t
        off = self.off[t]; p = self.cut_p[t]
        p1, p2, p4 = p[:, 0], p[:, 1], p[:, 2]
        hours = 0.5 * p1 + 0.5 * (p1 + p2) + (p2 + p4)                     # expected cut hours in the next 4 h
        if off.any():
            hours = np.where(off, 0.5 * (1 + p1) + 0.5 * (p1 + p2) + (p2 + p4), hours)
        load = self.load_fc[t].mean(1)
        usable = c.batt_kwh * (1 - c.soc_min) * c.eta
        need = min(float((c.essential_share * load * hours).sum()), usable)
        avail = max(self.soc - c.soc_min * c.batt_kwh, 0.0) * c.eta
        return need, avail

    def shield_cont(self, u, frac):
        c, t = self.cfg, self.t
        need, avail = self.need_avail()
        off = self.off[t]; P = c.batt_kw
        if u > 0:
            u = float(np.clip((avail - need) / (P * DT), 0.0, u))
        if frac > 0 and off.any():
            draw = (1 - c.essential_share) * self.demand[t][off].sum() * DT
            if draw > 1e-12:
                frac = float(np.clip((avail - need) / draw, 0.0, frac))
        hour = (t % SPD) / 4
        if avail < need and not off.any() and not (c.peak_hours[0] <= hour < c.peak_hours[1]):
            u = -1.0
        return u, frac

    def shield_action(self, a):
        c, t = self.cfg, self.t
        u, full = BATT_LEVELS[a % NL], a // NL
        need, avail = self.need_avail()
        off = self.off[t]; P = c.batt_kw
        while u > 0 and avail - u * P * DT < need:
            u -= 0.5
        if full and off.any():
            draw = (1 - c.essential_share) * self.demand[t][off].sum() * DT
            if avail - draw < need:
                full = 0
        hour = (t % SPD) / 4
        if avail < need and not off.any() and not (c.peak_hours[0] <= hour < c.peak_hours[1]):
            u = -1.0
        return int(np.where(np.isclose(BATT_LEVELS, u))[0][0]) + NL * int(full)

    # ---------- step ----------
    def step(self, action):
        a = int(action)
        if self.use_shield:
            b = self.shield_action(a); self.shield_steps += int(b != a); a = b
        return self._physics(BATT_LEVELS[a % NL], 1.0 if a // NL == 1 else 0.0)

    def step_cont(self, u, oth_frac, shield=None):
        if (self.use_shield if shield is None else shield):
            u2, f2 = self.shield_cont(u, oth_frac)
            self.shield_steps += int(abs(u2 - u) > 1e-9 or abs(f2 - oth_frac) > 1e-9); u, oth_frac = u2, f2
        return self._physics(u, oth_frac)

    def _physics(self, u, oth_frac):
        c, t = self.cfg, self.t
        u = float(np.clip(u, -1, 1))
        if abs(u) < 1e-9:
            u = 0.0
        d, off, pv = self.demand[t], self.off[t], self.pv[t]
        ess = c.essential_share * d * off
        oth = (1 - c.essential_share) * d * off * float(np.clip(oth_frac, 0, 1))
        e_room = max(self.soc - c.soc_min * c.batt_kwh, 0.0) * c.eta / DT
        P = c.batt_kw

        def serve(need, avail):
            tot = need.sum()
            if tot <= 0:
                return np.zeros_like(need), avail
            got = min(tot, avail)
            return need * (got / tot), avail - got
        ess_pv, pv_left = serve(ess, pv)
        oth_pv, pv_left = serve(oth, pv_left)
        b_avail = min(e_room, P) if u >= 0 else 0.0
        ess_b, b_left = serve(ess - ess_pv, b_avail)
        oth_b, b_left = serve(oth - oth_pv, b_left)
        b_out = b_avail - b_left
        ess_got, oth_got = ess_pv + ess_b, oth_pv + oth_b
        load_on = d[~off].sum()
        pv_on = min(pv_left, load_on); pv_surplus = pv_left - pv_on
        b_on = min(u * P, P - b_out, max(e_room - b_out, 0.0), load_on - pv_on) if u > 0 else 0.0
        room = c.batt_kwh - self.soc
        ch_pv = max(min(pv_surplus, P - b_out - b_on, room / (c.eta * DT)), 0.0)
        ch_grid = max(min(-u * P, P - ch_pv, room / (c.eta * DT) - ch_pv), 0.0) if u < 0 else 0.0
        dis, ch = b_out + b_on, ch_pv + ch_grid
        self.soc = float(np.clip(self.soc + (ch * c.eta - dis / c.eta) * DT, 0.0, c.batt_kwh))
        grid_import = load_on - pv_on - b_on + ch_grid
        ess_uns = (ess - ess_got) * DT
        oth_uns = ((1 - c.essential_share) * d * off - oth_got) * DT
        self.cut_demand += d * off * DT
        self.cut_served += (ess_got + oth_got) * DT
        hour = (t % SPD) / 4
        in_peak = c.peak_hours[0] <= hour < c.peak_hours[1]
        excess = max(grid_import - c.cap_kw, 0.0) if in_peak else 0.0
        r = -(c.w_ess * ess_uns.sum() + c.w_other * oth_uns.sum() + c.w_peak * grid_import * DT * in_peak
              + c.w_peak_excess * excess * DT + c.w_wear * dis * DT)
        L = self.log
        L["ess_uns"].append(ess_uns.sum()); L["oth_uns"].append(oth_uns.sum()); L["import"].append(grid_import)
        L["pv_used"].append((pv - max(pv_surplus - ch_pv, 0)) * DT); L["pv_avail"].append(pv * DT)
        L["dis"].append(dis * DT); L["ch"].append(ch * DT); L["soc"].append(self.soc); L["ess_uns_homes"].append(ess_uns > UNSERVED_KWH)
        self.t += 1; self.steps += 1
        done = self.t >= self.T
        if done:
            mask = self.cut_demand > 1e-6
            if mask.any():
                j = np.nan_to_num(jain(self.cut_served[mask] / self.cut_demand[mask]), nan=1.0)
                r += c.w_fair * (j - 1.0) * c.n_homes
        self.hist.append(self._obs_one())
        return self._stacked(), r * c.reward_scale, done, False, {}

    def metrics(self):
        c, L = self.cfg, self.log
        days = self.T / SPD
        imp = np.array(L["import"]).reshape(-1, SPD)[:, c.peak_hours[0] * 4:c.peak_hours[1] * 4]
        mask = self.cut_demand > 1e-6
        per = lambda x: float(x / c.n_homes / days * 30)
        return {
            "LOLH_ess_h_per_home_month": per(np.array(L["ess_uns_homes"]).sum() * DT),
            "ENS_ess_kwh_per_home_month": per(np.sum(L["ess_uns"])),
            "cut_demand_served_pct": float(100 * self.cut_served.sum() / max(self.cut_demand.sum(), 1e-9)),
            "jain_fairness": jain(self.cut_served[mask] / self.cut_demand[mask]) if mask.any() else 1.0,
            "evening_peak_import_kw": float(imp.max(axis=1).mean()),
            "evening_import_kwh_per_day": float(imp.sum() * DT / days),
            "evening_excess_kwh_per_day": float(np.clip(imp - c.cap_kw, 0, None).sum() * DT / days),
            "solar_used_pct": float(100 * np.sum(L["pv_used"]) / max(np.sum(L["pv_avail"]), 1e-9)),
            "battery_cycles_per_day": float(np.sum(L["dis"]) / max(c.batt_kwh, 1e-9) / days),
            "wear_cost_inr_per_home_month": per(np.sum(L["dis"]) * WEAR_INR_PER_KWH),
            "shield_rate_pct": 100.0 * self.shield_steps / max(self.steps, 1),
        }


def evaluate(policy, bank, episodes, cfg, env_kw, sb3=True):
    """Every episode at once (batched). policy = an SB3 model (deterministic) or a function (env, obs) -> action.
    Returns one row per episode (metrics and the reward per week)."""
    envs = [StreetEnv(bank, episodes, cfg, train=False, **env_kw) for _ in episodes]
    obs = np.stack([e.reset(options={"episode": i})[0] for i, e in enumerate(envs)])
    tot, active = np.zeros(len(envs)), np.ones(len(envs), bool)
    while active.any():
        act = policy.predict(obs, deterministic=True)[0] if sb3 else [policy(e, obs[i]) if active[i] else 0 for i, e in enumerate(envs)]
        for i in np.where(active)[0]:
            o, r, done, _, _ = envs[i].step(int(act[i])); tot[i] += r; obs[i] = o
            if done:
                active[i] = False
    return [{**e.metrics(), "reward_per_week": tot[i], "episode": i} for i, e in enumerate(envs)]

Writing ../src/sim.py


In [2]:
%%writefile ../src/planners.py
"""MILP planner (perfect foresight, MPC, MPC-CC). Same objective and physics as src/sim.py. Written from notebooks/03_simulator.ipynb.
Identical to the archived v1-v3 planner; the only addition is the optional reserve (chance constraint) used by MPC-CC."""
import numpy as np
import scipy.sparse as sp
from scipy.optimize import milp, LinearConstraint, Bounds
from sim import SPD, DT


def split_pv(pv, ess, oth, load_on):
    pv_e = np.minimum(pv, ess); left = pv - pv_e
    pv_o = np.minimum(left, oth); left = left - pv_o
    pv_n = np.minimum(left, load_on)
    return ess - pv_e, oth - pv_o, load_on - pv_n, left - pv_n, pv_o


def peak_mask(t, cfg):
    hour = (np.asarray(t) % SPD) / 4
    return ((hour >= cfg.peak_hours[0]) & (hour < cfg.peak_hours[1])).astype(float)


def solve_milp(nE, nO, lon, sur, peak, s0, cfg, terminal_value=0.0, force_no_grid_charge_first=False, n_binary=None,
               mip_rel_gap=1e-4, time_limit=None, reserve=None):
    """Per step: be, bo, bn (battery to essentials / other cut load / connected homes), cp, cg (charge from solar / grid),
    s (stored energy after the step), z (1 = grid charging allowed, then no discharge), x (import above cap in peak hours),
    q (reserve shortfall, kWh; only with `reserve`).
    reserve = (R_kwh, n_steps, w): s_h + q_h >= R_kwh for h < n_steps, each kWh of shortfall costing w x DT per step."""
    H = len(nE); c = cfg; K = 9; nv = K * H
    iv = lambda k, h: k * H + h
    cost = np.zeros(nv); w = c.w_wear * DT
    cost[iv(0, 0):iv(0, 0) + H] = -c.w_ess * DT + w
    cost[iv(1, 0):iv(1, 0) + H] = -c.w_other * DT + w
    cost[iv(2, 0):iv(2, 0) + H] = -c.w_peak * DT * peak + w
    cost[iv(3, 0):iv(3, 0) + H] = -1e-6
    cost[iv(4, 0):iv(4, 0) + H] = c.w_peak * DT * peak
    cost[iv(5, H - 1)] = -terminal_value
    cost[iv(7, 0):iv(7, 0) + H] = c.w_peak_excess * DT * peak
    r, cc, v, lo, hi = [], [], [], [], []; row = 0
    for h in range(H):
        for k in range(5):
            r.append(row); cc.append(iv(k, h)); v.append(1.0)
        lo.append(-np.inf); hi.append(c.batt_kw); row += 1
        r += [row, row]; cc += [iv(4, h), iv(6, h)]; v += [1.0, -c.batt_kw]; lo.append(-np.inf); hi.append(0.0); row += 1
        for k in range(3):
            r.append(row); cc.append(iv(k, h)); v.append(1.0)
        r.append(row); cc.append(iv(6, h)); v.append(c.batt_kw); lo.append(-np.inf); hi.append(c.batt_kw); row += 1
        for k, coef in [(0, DT / c.eta), (1, DT / c.eta), (2, DT / c.eta), (3, -DT * c.eta), (4, -DT * c.eta), (5, 1.0)]:
            r.append(row); cc.append(iv(k, h)); v.append(coef)
        if h > 0:
            r.append(row); cc.append(iv(5, h - 1)); v.append(-1.0)
        b = s0 if h == 0 else 0.0; lo.append(b); hi.append(b); row += 1
        r += [row, row, row]; cc += [iv(7, h), iv(2, h), iv(4, h)]; v += [-1.0, -1.0, 1.0]
        lo.append(-np.inf); hi.append(c.cap_kw - lon[h]); row += 1
    q_ub = np.zeros(H)
    if reserve is not None:
        R, n_res, w_q = reserve
        for h in range(min(n_res, H)):
            r += [row, row]; cc += [iv(5, h), iv(8, h)]; v += [1.0, 1.0]; lo.append(R); hi.append(np.inf); row += 1
            q_ub[h] = np.inf
        cost[iv(8, 0):iv(8, 0) + H] = w_q * DT
    A = sp.csr_matrix((v, (r, cc)), shape=(row, nv))
    zub = np.ones(H)
    if force_no_grid_charge_first:
        zub[0] = 0.0
    ub = np.concatenate([nE, nO, lon, sur, np.full(H, c.batt_kw), np.full(H, c.batt_kwh), zub, np.full(H, np.inf), q_ub])
    lb = np.concatenate([np.zeros(5 * H), np.full(H, c.soc_min * c.batt_kwh), np.zeros(3 * H)])
    nb = H if n_binary is None else min(n_binary, H)        # exact on/off only for the first n_binary steps
    integ = np.r_[np.zeros(6 * H), np.ones(nb), np.zeros(H - nb), np.zeros(2 * H)]
    res = milp(cost, constraints=LinearConstraint(A, lo, hi), integrality=integ, bounds=Bounds(lb, ub),
               options={"mip_rel_gap": mip_rel_gap, **({"time_limit": time_limit} if time_limit else {})})
    if res.x is None:
        raise RuntimeError(res.message)
    out = {k: res.x[iv(i, 0):iv(i, 0) + H] for i, k in enumerate(["be", "bo", "bn", "cp", "cg", "s", "z", "x", "q"])}
    out["obj"] = res.fun
    out["mip_gap"] = float(getattr(res, "mip_gap", 0.0) or 0.0); out["status"] = int(res.status)   # 1 = time limit hit
    return out


def to_action(plan, h, oth_need, pv_o, cfg):
    u = (plan["bn"][h] - plan["cg"][h]) / cfg.batt_kw
    return u, ((pv_o + plan["bo"][h]) / oth_need if oth_need > 1e-9 else 1.0)


def true_inputs(env, t0, H, cfg):
    t = np.arange(t0, t0 + H); d, off = env.demand[t], env.off[t]
    return (cfg.essential_share * (d * off).sum(1), (1 - cfg.essential_share) * (d * off).sum(1), (d * ~off).sum(1), env.pv[t])

Writing ../src/planners.py


In [3]:
%%writefile ../src/controllers.py
"""Baseline controllers and the expert. Written from notebooks/03_simulator.ipynb.
Rule policies, MPC v2 and perfect foresight are the archived v1-v3 controllers (baselines.py, baselines_v2.py; duration model off);
MPC-CC (chance-constrained MPC) is new (results/FINAL_SPEC.md section 6)."""
import numpy as np
from scipy.stats import norm
from sim import StreetEnv, BATT_LEVELS, NL, SPD
from planners import split_pv, peak_mask, solve_milp, to_action, true_inputs


# ---------- rules ----------
def act(level, full):
    return int(np.where(BATT_LEVELS == level)[0][0]) + (NL if full else 0)


def policy_idle_full(env, obs):
    return act(0.0, True)


def policy_backup_only(env, obs):
    return act(-1.0, True) if obs[0] < 0.999 and not obs[5] > 0 else act(0.0, True)


def policy_rule(env, obs):
    soc, hour = obs[0], (env.t % SPD) / 4
    full = soc > 0.25
    if 18 <= hour < 22 and soc > 0.40:
        return act(1.0, full)
    if hour < 6 and soc < 0.90 and obs[5] == 0:
        return act(-1.0, full)
    return act(0.0, full)


POLICIES = {"idle_full": policy_idle_full, "backup_only": policy_backup_only, "rule": policy_rule}


def run_policy(bank, eps, cfg, name, i, shield=False, **kw):
    env = StreetEnv(bank, eps, cfg, train=False, shield=shield, **kw)
    obs, _ = env.reset(options={"episode": i}); done, tot = False, 0.0
    while not done:
        obs, r, done, _, _ = env.step(POLICIES[name](env, obs)); tot += r
    return {**env.metrics(), "reward_per_week": tot, "episode": i}


# ---------- perfect foresight (bound) ----------
def run_perfect(bank, eps, cfg, i, time_limit=600):
    env = StreetEnv(bank, eps, cfg, train=False, shield=False); env.reset(options={"episode": i})
    ess, oth, lon, pv = true_inputs(env, 0, env.T, cfg); nE, nO, nL, sur, pv_o = split_pv(pv, ess, oth, lon)
    plan = solve_milp(nE, nO, nL, sur, peak_mask(np.arange(env.T), cfg), env.soc, cfg, time_limit=time_limit)
    done, tot = False, 0.0
    while not done:
        t = env.t; _, r, done, _, _ = env.step_cont(*to_action(plan, t, oth[t], pv_o[t], cfg)); tot += r
    return {**env.metrics(), "reward_per_week": tot, "episode": i, "pf_mip_gap": plan["mip_gap"], "pf_time_limited": plan["status"] == 1}


# ---------- MPC ----------
def _interp(h, pts, vals):
    return np.array([np.interp(h, pts, vals[:, j]) for j in range(vals.shape[1])])


def mpc_inputs(env, t, H, cfg):
    """Expected street inputs for steps t .. t+H-1 from information available at t (cut and demand forecasts up to +4 h,
    then the 7-day same-time cut rate and yesterday's demand; solar = yesterday, now = last completed hour)."""
    idx = t + np.arange(H); idx_c = np.minimum(idx, env.T - 1)
    past = idx_c - SPD; have_y = past >= 0
    d_now, off_now = env.demand[t], env.off[t]
    d = np.where(have_y[:, None], env.demand[np.maximum(past, 0)], d_now[None, :])
    pv = np.where(have_y, env.pv[np.maximum(past, 0)], env.pv_prev[idx_c] * cfg.pv_kwp)
    p = np.repeat(env.cut_rate[(t // SPD) * SPD + idx % SPD][:, None], len(d_now), 1)
    pts = np.array([0, 4, 8, 16])
    P = np.vstack([off_now.astype(float), env.cut_p[t].T]); L = np.vstack([d_now, env.load_fc[t].T])
    for h in range(1, min(17, H)):
        p[h] = _interp(h, pts, P); d[h] = _interp(h, pts, L)
    p[0] = off_now.astype(float); d[0] = d_now
    pv[0] = env.pv_obs[t] * cfg.pv_kwp
    ess = cfg.essential_share * (p * d).sum(1); oth = (1 - cfg.essential_share) * (p * d).sum(1); lon = ((1 - p) * d).sum(1)
    return ess, oth, lon, pv, idx


def reserve_kwh(env, z):
    """MPC-CC reserve on stored energy: R = need_mean + z * need_sd (in usable energy), as stored kWh above the floor."""
    c, t = env.cfg, env.t
    need, _ = env.need_avail()
    off = env.off[t]; p = env.cut_p[t]; p1, p2, p4 = p[:, 0], p[:, 1], p[:, 2]
    hours = np.where(off, 0.5 * (1 + p1) + 0.5 * (p1 + p2) + (p2 + p4), 0.5 * p1 + 0.5 * (p1 + p2) + (p2 + p4))
    load = env.load_fc[t].mean(1)
    sd = float(np.sqrt(((c.essential_share * load * hours) ** 2 * p1 * (1 - p1)).sum()))
    usable = c.batt_kwh * (1 - c.soc_min) * c.eta
    R = min(need + z * sd, usable)
    return c.soc_min * c.batt_kwh + R / c.eta


def run_mpc(bank, eps, cfg, i, kind="v2", eps_cc=None, H=96):
    """kind "v2": MPC v2 (forecasts, shield on). kind "cc": MPC-CC (forecasts, reserve with risk eps_cc, no shield)."""
    env = StreetEnv(bank, eps, cfg, train=False, shield=(kind == "v2")); env.reset(options={"episode": i})
    z = norm.ppf(1 - eps_cc) if kind == "cc" else None
    done, tot = False, 0.0
    while not done:
        t = env.t
        ess, oth, lon, pv, idx = mpc_inputs(env, t, H, cfg)
        nE, nO, nL, sur, pv_o = split_pv(pv, ess, oth, lon)
        cut_now = bool(env.off[t].any())
        res = (reserve_kwh(env, z), 16, cfg.w_ess) if kind == "cc" else None
        plan = solve_milp(nE, nO, nL, sur, peak_mask(idx, cfg), env.soc, cfg, terminal_value=cfg.w_peak,
                          force_no_grid_charge_first=cut_now, mip_rel_gap=1e-2, reserve=res)
        u, frac = to_action(plan, 0, oth[0], pv_o[0], cfg)
        if cut_now and u < 0:
            u = 0.0
        _, r, done, _, _ = env.step_cont(u, frac); tot += r
    return {**env.metrics(), "reward_per_week": tot, "episode": i}


# ---------- the expert (perfect foresight re-planned from the state reached) ----------
def expert_plan(env, t, soc, cfg):
    ess, oth, lon, pv = true_inputs(env, t, env.T - t, cfg)
    nE, nO, nL, sur, pv_o = split_pv(pv, ess, oth, lon)
    plan = solve_milp(nE, nO, nL, sur, peak_mask(np.arange(t, env.T), cfg), soc, cfg, n_binary=32, mip_rel_gap=1e-3)
    return plan, oth, pv_o


def discrete(u, frac):
    return int(np.argmin(np.abs(BATT_LEVELS - u))) + NL * int(frac >= 0.5)


def demonstrate(bank, eps, cfg, i, replan=16):
    """Expert on one episode (no shield), re-planning every `replan` steps. Returns metrics, the 23-input states and actions."""
    env = StreetEnv(bank, eps, cfg, train=False, shield=False)
    obs, _ = env.reset(options={"episode": i}); done, tot, O, A = False, 0.0, [], []
    while not done:
        t = env.t
        if t % replan == 0:
            plan, oth, pv_o = expert_plan(env, t, env.soc, cfg); t0 = t
        h = t - t0
        a = discrete(*to_action(plan, h, oth[h], pv_o[h], cfg))
        O.append(obs); A.append(a)
        obs, r, done, _, _ = env.step(a); tot += r
    return {**env.metrics(), "reward_per_week": tot, "episode": i}, np.asarray(O, np.float32), np.asarray(A, np.int64)


def expert_label(env, t, soc, cfg):
    plan, oth, pv_o = expert_plan(env, t, soc, cfg)
    return discrete(*to_action(plan, 0, oth[0], pv_o[0], cfg))

Writing ../src/controllers.py


In [4]:
import sys, os, json, hashlib, pickle, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src")); os.environ["PYTHONPATH"] = str(ROOT / "src")
PROC, RES, A = ROOT / "data" / "processed", ROOT / "results", ROOT / "_archive"
assert hashlib.sha256((RES / "FINAL_SPEC.md").read_bytes()).hexdigest() == json.loads((RES / "FINAL_SPEC_lock.json").read_text())["sha256"], "plan changed"
split = json.loads((PROC / "split.json").read_text())
assert hashlib.sha256(json.dumps(split, indent=1).encode()).hexdigest() == json.loads((PROC / "split_lock.json").read_text())["sha256"], "split changed"
gate = json.loads((RES / "forecast_gate.json").read_text()); blob = (ROOT / "models" / "forecasters.pkl").read_bytes()
assert hashlib.sha256(blob).hexdigest() == gate["sha256"], "forecasters changed after freeze"
F = pickle.loads(blob)
# system and reward: unchanged from versions 1-3
cfg_old = json.loads((A / "results_v1-v4" / "step5_config.json").read_text())
(RES / "config.json").write_text(json.dumps({"config": cfg_old["config"], "peak_cap_rule": cfg_old["peak_cap_rule"], "wear": cfg_old["wear"],
                                             "note": "identical to versions 1-3 (results/step5_config.json in the archive)"}, indent=1))
import sim, planners, controllers, features
from sim import StreetEnv, load_cfg, evaluate, N_OBS
from controllers import run_policy, run_perfect, run_mpc, demonstrate
from features import forecasts, make_bank_region, cut_table, runs_into_since, H, SPD
CFG = load_cfg(ROOT)
print("plan, split and forecasters verified | gate:", gate["gate"], "| config:", CFG)

plan, split and forecasters verified | gate: {'cut': True, 'load': True} | config: StreetConfig(n_homes=10, pv_kwp_per_home=0.5, batt_kwh_per_home=0.75, batt_kw_per_home=0.25, eta=0.95, soc_min=0.1, essential_share=0.3, peak_hours=(18, 22), peak_cap_kw_per_home=0.576, w_ess=10.0, w_other=3.0, w_peak=1.0, w_peak_excess=3.0, w_wear=0.2, w_fair=5.0, reward_scale=0.1, use_forecasts=True)


## 1. Regression check against the archived version 2 code and results (validation episodes)

In [5]:
old = pickle.loads((A / "data_processed" / "v2" / "episodes_v2_trainval.pkl").read_bytes())["val"]
ob = {f"ep{i}": {"start": np.datetime64(pd.Timestamp(e["start"]), "m"), "demand": e["demand"], "cut": e["cut"], "pv": e["pv"],
                 "pv_obs": e["pv_obs"], "pv_prev": e["pv_prev"], "cut_rate": e["cut_rate"], "cut_p": e["cut_p"], "load_fc": e["load_fc"]}
      for i, e in enumerate(old)}
oe = [{"region": f"ep{i}", "t0": 0, "pool_idx": list(range(len(e["pool"]))), "homes_idx": [e["pool"].index(h) for h in e["homes"]]} for i, e in enumerate(old)]
base = pd.read_csv(A / "results_v1-v4" / "v2" / "val_baselines_v2_per_episode.csv")
ref = lambda m: base[base.Method == m].set_index("episode").reward_per_week
chk = {}
new = pd.DataFrame([run_policy(ob, oe, CFG, "rule", i) for i in range(len(oe))]).set_index("episode").reward_per_week
chk["rule-based (116 weeks)"] = float((new - ref("3 Rule-based")).abs().max())
new = pd.DataFrame([run_policy(ob, oe, CFG, "rule", i, shield=True) for i in range(len(oe))]).set_index("episode").reward_per_week
chk["rule-based + shield (116 weeks)"] = float((new - ref("3s Rule-based + shield")).abs().max())
few = [0, 29, 58, 87]
chk["perfect foresight (4 weeks)"] = float(max(abs(run_perfect(ob, oe, CFG, i, time_limit=None)["reward_per_week"] - ref("5 Perfect foresight")[i]) for i in few))
chk["MPC v2 (2 weeks)"] = float(max(abs(run_mpc(ob, oe, CFG, i, "v2")["reward_per_week"] - ref("4c MPC v2 (risk-aware)")[i]) for i in few[:2]))
dem = pd.read_csv(A / "results_v1-v4" / "v2" / "demonstrator_val_per_episode.csv").set_index("episode").reward_per_week
chk["expert demonstrator (2 weeks)"] = float(max(abs(demonstrate(ob, oe, CFG, i)[0]["reward_per_week"] - dem[i]) for i in few[:2]))
from stable_baselines3 import PPO
m = PPO.load(ROOT / "models" / "reference" / "ppo_bc_v3" / "PPO-BC_s0.zip", device="cpu")
new = pd.DataFrame(evaluate(m, ob, oe, CFG, dict(shield=True, stack=1))).set_index("episode").reward_per_week
arc = pd.read_csv(A / "models_v1-v4" / "v2" / "rl" / "V2-PPO-BC_s0" / "best_val_per_episode.csv").set_index("episode").reward_per_week
chk["frozen PPO-BC seed 0 (116 weeks)"] = float((new - arc).abs().max())
display(pd.Series(chk, name="largest |new - archived| reward/week").to_frame())
assert all(v < 1e-6 for v in chk.values()), "the new code does not reproduce the archived results"
print("OK: simulator, shield, MILP planners, expert and policy evaluation reproduce the archived v2 results")
del old, ob

,largest |new - archived| reward/week
rule-based (116 weeks),7.105427e-15
rule-based + shield (116 weeks),7.105427e-15
perfect foresight (4 weeks),7.105427e-15
MPC v2 (2 weeks),0.000000e+00
expert demonstrator (2 weeks),0.000000e+00
frozen PPO-BC seed 0 (116 weeks),3.552714e-15


OK: simulator, shield, MILP planners, expert and policy evaluation reproduce the archived v2 results


## 2. Simulator bank (training and validation regions) and episode lists
Forecast inputs come from the frozen forecasters (notebook 02). A forecaster that failed its gate is replaced by its best simple baseline.

In [6]:
def baseline_cut_p(tab):
    into, since = runs_into_since(tab["cut"]); P = []
    for hn, h in H.items():
        X = cut_table(tab["cut"], h, into, since)
        P.append(pd.Series(X.home_slot7.fillna(X.city_slot7).fillna(0).clip(0, 1).values, index=X.index).unstack().reindex(index=tab["cut"].index, columns=tab["cut"].columns))
    return np.nan_to_num(np.stack([p.values for p in P], -1), nan=0.0).astype(np.float32)
def baseline_load(tab):
    d = tab["demand"]
    return np.nan_to_num(np.stack([d.shift(SPD - h).values for h in H.values()], -1), nan=0.0).astype(np.float32)
t0 = time.time(); BANK = {}
for r in ["Mathura", "Bareilly", "UP"]:
    tab = {k: pd.read_parquet(PROC / f"{r}_{k}.parquet") for k in ["cut", "demand", "supplied_load"]}
    tab["solar"] = pd.read_parquet(PROC / f"{r}_solar_15min.parquet"); tab["hourly"] = pd.read_parquet(PROC / f"{r}_solar_hourly_causal.parquet")
    cp, lf = forecasts(tab, F, split["scalers"])
    if not gate["gate"]["cut"]: cp = baseline_cut_p(tab)
    if not gate["gate"]["load"]: lf = baseline_load(tab)
    BANK[r] = make_bank_region(tab, cp, lf)
    print(f"{r}: {BANK[r]['demand'].shape} ({time.time() - t0:.0f}s)")
(PROC / "bank_trainval.pkl").write_bytes(pickle.dumps(BANK))

def ep_of(w):
    B = BANK[w["region"]]; t0 = int((np.datetime64(pd.Timestamp(w["start"]), "m") - B["start"]) / np.timedelta64(15, "m"))
    e = {"region": w["region"], "start": w["start"], "t0": t0, "pool_idx": sorted(B["homes"].index(h) for h in w["pool"])}
    if "homes" in w: e["homes_idx"] = sorted(B["homes"].index(h) for h in w["homes"])
    if "hold" in w: e["hold"] = w["hold"]
    return e
E = {"train": [ep_of(w) for w in split["train"]], "val": [ep_of(w) for w in split["val"]]}
(PROC / "episodes.json").write_text(json.dumps(E))
# the bank reproduces the region tables for every episode (spot check) and the archived Mathura inputs
dem = pd.read_parquet(PROC / "Mathura_demand.parquet")
for e, w in zip(E["val"], split["val"]):
    if w["region"] != "Mathura": continue
    ix = pd.date_range(w["start"], periods=672, freq="15min")
    assert np.array_equal(BANK["Mathura"]["demand"][e["t0"]:e["t0"] + 672][:, e["homes_idx"]], dem.reindex(ix)[sorted(w["homes"], key=BANK["Mathura"]["homes"].index)].values.astype(np.float32))
print({k: len(v) for k, v in E.items()}, "| train windows per region:", pd.Series([e["region"] for e in E["train"]]).value_counts().to_dict(),
      "| validation weeks per region:", pd.Series([e["region"] for e in E["val"]]).value_counts().to_dict())

Mathura: (63552, 38) (62s)


Bareilly: (87072, 46) (171s)


UP: (35040, 36) (207s)


{'train': 980, 'val': 73} | train windows per region: {'Bareilly': 486, 'Mathura': 374, 'UP': 120} | validation weeks per region: {'Bareilly': 43, 'Mathura': 17, 'UP': 13}


## 3. Checks: Gymnasium API, observation ranges, shield activity (random actions, train and validation)

In [7]:
from gymnasium.utils.env_checker import check_env
for st in (1, 8):
    check_env(StreetEnv(BANK, E["train"], CFG, shield=True, stack=st), skip_render_check=True)
print("check_env passed (state 23 and 8 x 23)")
rows = []
for part in ["train", "val"]:
    env = StreetEnv(BANK, E[part], CFG, train=(part == "train"), seed=0, shield=True); rng = np.random.default_rng(0); O = []
    for i in range(0, len(E[part]), max(1, len(E[part]) // 25)):
        o, _ = env.reset(options={"episode": i}); done = False
        while not done:
            o, r, done, _, _ = env.step(int(rng.integers(10))); O.append(o)
    O = np.array(O); rows.append({"part": part, "min": O.min(), "max": O.max(), "finite": bool(np.isfinite(O).all()), "shield changed (%)": env.metrics()["shield_rate_pct"]})
display(pd.DataFrame(rows))
assert all(r["finite"] and r["min"] >= -1 and r["max"] <= 5 for r in rows)

check_env passed (state 23 and 8 x 23)


,part,min,max,finite,shield changed (%)
0,train,-1.0,1.5388,True,6.845238
1,val,-1.0,1.5176,True,2.529762
